In [2]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import warnings
warnings.filterwarnings("ignore")

In [24]:
train_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
val_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
test_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"

# load train,test & validation data
train_data = pd.read_parquet(train_path)
# val_data = pd.read_parquet(val_path)
test_data = pd.read_parquet(test_path)

In [30]:
train_data.head()

,code,generator,label,language
0,"(a, b, c, d) = [int(x) for x in input().split(...",human,0,Python
1,valid version for the language; all others can...,Qwen/Qwen2.5-Coder-1.5B,1,Python
2,python\ndef min_cards_to_flip(s):\n vowels ...,Qwen/Qwen2.5-Coder-7B-Instruct,1,Python
3,T = int(input())\nfor t in range(T):\n\tcolor ...,human,0,Python
4,for i in range(int(input())):\n\tinput()\n\ta ...,human,0,Python


In [31]:


# Basic cleaning
# ====================================================
#  Extra features 
# ====================================================

camel_case_pattern = r'\b[a-z]+(?:[A-Z][a-z0-9]+)+\b'
snake_case_pattern = r'\b[a-z]+(?:_[a-z0-9]+)+\b'

def count_camel(text):
    return len(re.findall(camel_case_pattern, text))

def count_snake(text):
    return len(re.findall(snake_case_pattern, text))


# training dataset

train_data['code'] = train_data['code'].astype(str)

train_data['camel_count'] = train_data['code'].apply(count_camel)
train_data['snake_count'] = train_data['code'].apply(count_snake)

train_data['total_idents'] = train_data['camel_count'] + train_data['snake_count']
train_data['camel_ratio'] = train_data['camel_count'] / (train_data['total_idents'] + 1e-5)
train_data['snake_ratio'] = train_data['snake_count'] / (train_data['total_idents'] + 1e-5)


# Code Features Extraction 
train_data["code_length"] = train_data["code"].str.len()
train_data['num_comments'] = train_data['code'].str.count("#|//")
train_data['num_tabs'] = train_data['code'].str.count("\t")
train_data['num_spaces'] = train_data['code'].str.count("    ")

train_data["num_lines"] = train_data["code"].apply(lambda x: x.count("\n") + 1)
train_data["num_defs"] = train_data["code"].str.count(r"\bdef\b")
train_data["num_for"] = train_data["code"].str.count(r"\bfor\b")
train_data["num_if"] = train_data["code"].str.count(r"\bif\b")
train_data["num_import"] = train_data["code"].str.count(r"\bimport\b")


In [32]:
# test dataset 

test_data['code'] = test_data['code'].astype(str)

test_data['camel_count'] = test_data['code'].apply(count_camel)
test_data['snake_count'] = test_data['code'].apply(count_snake)
test_data['total_idents'] = test_data['camel_count'] + test_data['snake_count']
test_data['camel_ratio'] = test_data['camel_count'] / (test_data['total_idents'] + 1e-5)
test_data['snake_ratio'] = test_data['snake_count'] / (test_data['total_idents'] + 1e-5)

# Code Features Extraction 
test_data["code_length"] = test_data["code"].str.len()
test_data['num_comments'] = test_data['code'].str.count("#|//")
test_data['num_tabs'] = test_data['code'].str.count("\t")
test_data['num_spaces'] = test_data['code'].str.count("    ")

test_data["num_lines"] = test_data["code"].apply(lambda x: x.count("\n") + 1)
test_data["num_defs"] = test_data["code"].str.count(r"\bdef\b")
test_data["num_for"] = test_data["code"].str.count(r"\bfor\b")
test_data["num_if"] = test_data["code"].str.count(r"\bif\b")
test_data["num_import"] = test_data["code"].str.count(r"\bimport\b")

In [ ]:
# # ====================================================
# # 3. Label encoding for generator & language
# # ====================================================

# for col in ['generator', 'language']:
#     le = LabelEncoder()
#     train_data[col] = le.fit_transform(train_data[col].astype(str))

In [33]:
# ====================================================
# Train-test split
# ====================================================

X_train_text, y_train = train_data["code"], train_data["label"]
X_test_text, y_test = test_data["code"], test_data["label"]


In [35]:
# ====================================================
# TF-IDF vectorization
# ====================================================

tfidf = TfidfVectorizer(
    strip_accents='unicode',
    analyzer='word',
    # ngram_range=(1,2),
    max_features=20000
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

In [ ]:

# ====================================================
#  Define Models
# ====================================================

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Multinomial NB": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier()
}

# Optional (if installed)
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        eval_metric='logloss',
        tree_method='hist'
    )
except:
    pass

try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier()
except:
    pass

# ====================================================
# Train and Evaluate All Models
# ====================================================

results = []

for name, model in models.items():
    print("="*60)
    print(f"Training {name}...")
    
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(classification_report(y_test, pred))

    results.append([name, acc, f1])

# Results summary
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score"])
results_df = results_df.sort_values("F1 Score", ascending=False)

print("\n\n==== FINAL RESULTS ====")
print(results_df)


Training Logistic Regression...
              precision    recall  f1-score   support

           0       0.91      0.28      0.43       777
           1       0.27      0.91      0.41       223

    accuracy                           0.42      1000
   macro avg       0.59      0.59      0.42      1000
weighted avg       0.77      0.42      0.43      1000

Training Multinomial NB...
              precision    recall  f1-score   support

           0       0.87      0.15      0.26       777
           1       0.24      0.92      0.38       223

    accuracy                           0.33      1000
   macro avg       0.55      0.54      0.32      1000
weighted avg       0.73      0.33      0.29      1000

Training Linear SVM...
              precision    recall  f1-score   support

           0       0.93      0.29      0.44       777
           1       0.27      0.92      0.42       223

    accuracy                           0.43      1000
   macro avg       0.60      0.60      0.43   

 ## With Metadata features

In [50]:

# ====================================================
#  Metadata features
# ====================================================

meta_features = [
    # 'language',
    'camel_count',
    'snake_count',
    'camel_ratio',
    'snake_ratio',
    "code_length",
    "num_comments",
    "num_tabs",
    "num_spaces",
    "num_lines",
    "num_defs",
    "num_for",
    "num_if",
    "num_import"   
]

X_train_meta = train_data[meta_features].values
X_test_meta = test_data[meta_features].values


# Combine sparse and dense features
from scipy.sparse import hstack

X_train = hstack([X_train_tfidf, X_train_meta])
X_test  = hstack([X_test_tfidf, X_test_meta])

In [ ]:
results = []

for name, model in models.items():
    print("="*60)
    print(f"Training {name}...")
    
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    print(classification_report(y_test, pred))

    results.append([name, acc, f1])

# Results summary
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score"])
results_df = results_df.sort_values("F1 Score", ascending=False)